# Genex Interview + Activity Brain — v4

This notebook is the cleaned working notebook for the current Genex 0–5 developmental interview and activity-planning engine.

It supports two interview modes that share the same downstream planning pipeline:

## Mode A — Live parent mode
We answer the milestone questions directly, one by one, as if we were the parent or evaluator.

## Mode B — Synthetic parent mode
Gemini simulates the parent answers one question at a time from the child profile, while the Genex interview engine still selects the questions and computes the developmental profile.

After either mode, the notebook runs the same shared downstream steps:

1. delay estimation by domain
2. milestone question interview by category
3. developmental-age estimate by category
4. support-tier assignment by category
5. category activity-bank generation
6. weekly schedule generation
7. case summary table
8. optional advisor-report export

## Important working rule

If we edit any function cell, we should **restart the kernel and run the notebook from top to bottom again**.  
Most of the crashes in earlier versions were not caused by the file itself; they happened because the kernel had only some of the updated function definitions loaded.


## Recommended run order

1. Install cell  
2. Imports + environment setup  
3. Config + CDC loading  
4. Core engine cells  
5. Choose one mode:
   - run one live case
   - run one synthetic case
   - run a full advisor export packet

We strongly recommend using **Run All** or **Run All Above** before running the example cells at the end.


In [21]:
# Install once per notebook kernel if needed
%pip install -U pandas openpyxl python-dotenv openai google-genai reportlab



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install -U google-genai


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
os.environ["GEMINI_API_KEY"] = ""
print("GEMINI_API_KEY visible:", bool(os.environ.get("GEMINI_API_KEY")))

GEMINI_API_KEY visible: True


In [24]:
# ------------------------------------------------------------
# Imports + environment setup
# ------------------------------------------------------------
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    from google import genai
except ImportError:
    genai = None

from reportlab.lib.pagesizes import landscape, LETTER
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from reportlab.pdfbase.pdfmetrics import stringWidth

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "genex_interview_activity_v4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY) if (OpenAI is not None and OPENAI_API_KEY) else None
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if (genai is not None and GEMINI_API_KEY) else None

print("OpenAI available:", openai_client is not None)
print("Gemini available:", gemini_client is not None)

# Optional temporary setup if needed for debugging.
# Do NOT save real keys inside the notebook file.
#
# import os
# os.environ["OPENAI_API_KEY"] = "your_real_openai_key_here"
# os.environ["GEMINI_API_KEY"] = "your_real_gemini_key_here"
# OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
# openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
# gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None


OpenAI available: True
Gemini available: True


In [25]:
# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotional": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",

    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",
    "social_emotional": "social_and_emotional",
    "social": "social_and_emotional",

    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",

    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.4,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())
VALID_PARENT_ANSWERS = {"yes", "sometimes", "with_help", "no", "not_sure"}


In [26]:
# ------------------------------------------------------------
# CDC loading + normalization
# ------------------------------------------------------------
def find_cdc_file(filename: str = "milestone-cdc-table.xlsx") -> Path:
    """Find the CDC milestone table in common project locations."""
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the repo root or notebooks parent folder."
    )

def load_cdc_table(path: Optional[str] = None):
    """Load and normalize the CDC milestone table."""
    path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(path)

    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [
        f"{row.category_key}_{row.months}_{i}"
        for i, row in enumerate(df.itertuples(), start=1)
    ]
    return df, path

cdc_df, cdc_path = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())

print("Loaded CDC file:", cdc_path)
print("CDC ages:", CDC_AGES)
print("Rows:", len(cdc_df))

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

display(cdc_df.head())


Loaded CDC file: /Users/sara/Projects/Genex/milestone-cdc-table.xlsx
CDC ages: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows: 159


,months,category,milestone,category_key,question_id
0,2,social and emotial,calms down when spoken to or picked up,social_and_emotional,social_and_emotional_2_1
1,2,social and emotial,looks at your face,social_and_emotional,social_and_emotional_2_2
2,2,social and emotial,seems happy to see you when you walk up to her,social_and_emotional,social_and_emotional_2_3
3,2,social and emotial,smiles when you talk to or smile at her,social_and_emotional,social_and_emotional_2_4
4,2,language and communication,makes sounds other than crying,language_and_communication,language_and_communication_2_5


In [27]:
# ------------------------------------------------------------
# Core helpers + state
# ------------------------------------------------------------
def normalize_answer(answer_text: str) -> str:
    """Normalize a raw parent answer into the allowed answer set."""
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def extract_json_block(text: str) -> dict:
    """Parse strict JSON or recover a simple answer from loosely formatted model text."""
    text = str(text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    lowered = text.lower()
    for label in ["with_help", "not_sure", "sometimes", "yes", "no"]:
        if label in lowered:
            return {"answer": label, "reason": text[:200]}

    if "with help" in lowered:
        return {"answer": "with_help", "reason": text[:200]}
    if "not sure" in lowered or "unsure" in lowered or "maybe" in lowered:
        return {"answer": "not_sure", "reason": text[:200]}

    raise ValueError(f"Could not parse JSON from model output: {text[:300]}")

def new_state() -> Dict[str, Any]:
    """Initialize the working state dictionary for one case."""
    return {
        "child": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "activity_banks": {},
        "weekly_slot_allocation": {},
        "weekly_schedule": {},
    }

def print_child_reference(state: Dict[str, Any]) -> None:
    child = state["child"]
    print("\n" + "=" * 100)
    print("CHILD REFERENCE CHECK")
    print("=" * 100)
    print(f"Name: {child.get('name', '')}")
    print(f"Chronological age: {child.get('chronological_months', '')} months")
    print(f"Diagnosis / condition: {child.get('diagnosis', '')}")
    print(f"Parent concern: {child.get('concern', '')}")
    print(f"Daily time available: {child.get('daily_time_min', '')} minutes")

def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    """Collect a live child profile from user input."""
    print_banner("INTAKE AGENT")

    name = input("What is your child's name? ").strip()
    chronological_months = int(
        input("How old is your child in months? (for example: 18, 24, 36, 48, 60) ").strip()
    )
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()
    daily_time_min = int(
        input("About how many minutes per day can you usually spend helping or playing with your child? ").strip()
    )

    state["child"] = {
        "name": name,
        "chronological_months": chronological_months,
        "diagnosis": diagnosis,
        "concern": concern,
        "daily_time_min": daily_time_min,
    }
    return state["child"]

def init_state_from_case(case: Dict[str, Any], daily_time_min: int = 10) -> Dict[str, Any]:
    """Initialize a state object from a prefilled synthetic case."""
    state = new_state()
    state["child"] = {
        "name": case["child_name"],
        "chronological_months": int(case["age_months"]),
        "diagnosis": case["diagnosis"],
        "concern": case["concerns"],
        "daily_time_min": int(daily_time_min),
    }
    return state


In [28]:
# ------------------------------------------------------------
# Delay estimation
# ------------------------------------------------------------
def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    """Estimate a rough starting delay in months for one domain.

    This is only a starting anchor for question selection, not the final developmental age.
    """
    category_display = DOMAIN_CONFIG[category_key]["display"]
    concern_l = (concern or "").lower()

    fallback_by_category = {
        "movement_and_physical": 3,
        "language_and_communication": 3,
        "social_and_emotional": 6,
        "cognitive": 6,
    }

    domain_keywords = {
        "movement_and_physical": [
            "motor", "movement", "walk", "run", "jump", "balance", "coordination",
            "fine motor", "gross motor", "grasp", "hand", "writing", "stairs", "falls",
            "hypotonia", "sitting", "rolling", "crawling"
        ],
        "language_and_communication": [
            "speech", "language", "talk", "communication", "words", "sentence",
            "understand", "expressive", "receptive", "verbal", "babbling", "no words"
        ],
        "social_and_emotional": [
            "social", "peer", "friend", "play", "emotion", "emotional",
            "behavior", "anger", "meltdown", "interaction", "turn taking",
            "regulation", "eye contact", "transitions", "bullied", "bullying"
        ],
        "cognitive": [
            "attention", "focus", "concentration", "school", "learning", "routine",
            "executive", "task", "independent", "adaptive", "toilet", "dressing",
            "self-care", "directions", "paying attention"
        ],
    }

    has_domain_signal = any(kw in concern_l for kw in domain_keywords.get(category_key, []))

    if openai_client is None:
        delay_months = fallback_by_category.get(category_key, 6)
        if not has_domain_signal:
            delay_months = min(delay_months, 6)
        return {
            "delay_months": delay_months,
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    prompt = f"""
You are a pediatric developmental delay estimator agent for children ages 0 to 5 years.

Your job is to estimate a SINGLE STARTING DELAY in months for one developmental domain only.

This is NOT a diagnosis.
This is NOT a final developmental age.
This is ONLY a rough starting anchor for question selection.

Definition:
delay_months = chronological age in months - estimated functional developmental age in this specific domain

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Instructions:
1. Think only about THIS domain, not overall development.
2. If the diagnosis or concern does NOT meaningfully affect this domain, return 0 to 6 months.
3. If this domain IS affected, estimate the child's functional developmental level in this domain, then convert it into delay_months.
4. Be conservative but realistic.
5. Never exceed the child's chronological age.
6. Return only one integer number of months.
7. Return strict JSON only.

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )

        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(parsed.get("delay_months", fallback_by_category.get(category_key, 6)))
        delay_months = max(0, min(delay_months, chronological_months))

        if not has_domain_signal and delay_months > 6:
            delay_months = 6

        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }

    except Exception as e:
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    """Estimate a starting delay for each category."""
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]


In [29]:
# ------------------------------------------------------------
# Core milestone engine
# ------------------------------------------------------------
def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 3,
    max_bands: int = 3,
    max_questions_total: int = 9,
) -> List[Dict[str, Any]]:
    """Build a focused set of milestone questions around the estimated developmental range."""
    child = state["child"]
    chrono_months = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    available_months = sorted(subset["months"].unique().tolist())

    ordered_months = sorted(
        available_months,
        key=lambda m: (abs(m - approx_dev_months), m)
    )[:max_bands]

    questions = []
    for month in ordered_months:
        month_rows = subset[subset["months"] == month].sort_values("milestone")
        month_rows = month_rows.head(target_questions_per_band)

        for _, row in month_rows.iterrows():
            questions.append({
                "question_id": row["question_id"],
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
            })

    return questions[:max_questions_total]

def compute_dev_age_from_answers(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> int:
    """Estimate developmental age using conservative month-band confirmation.

    A month band is confirmed only if:
    - it has at least `min_yes_confirm` YES answers
    - and YES answers are at least `yes_ratio_confirm` of that band
    """
    if not answers:
        return 6

    band_summary = {}
    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {"total": 0, "yes": 0, "partial": 0, "no": 0}

        band_summary[month]["total"] += 1

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    answered_months = sorted(band_summary.keys())
    confirmed_months = []
    partial_months = []

    for month in answered_months:
        total = band_summary[month]["total"]
        yes_count = band_summary[month]["yes"]
        partial_count = band_summary[month]["partial"]
        yes_ratio = yes_count / total if total > 0 else 0

        if yes_count >= min_yes_confirm and yes_ratio >= yes_ratio_confirm:
            confirmed_months.append(month)
        elif yes_count > 0 or partial_count > 0:
            partial_months.append(month)

    if confirmed_months:
        return int(max(confirmed_months))

    if partial_months:
        anchor = int(min(partial_months))
        lower_candidates = [m for m in answered_months if m < anchor]
        return int(max(lower_candidates)) if lower_candidates else int(anchor)

    return int(min(answered_months))

def summarize_answers_by_band(answers: List[Dict[str, Any]]) -> Dict[int, Dict[str, Any]]:
    """Summarize answers by month band for debugging and advisor review."""
    band_summary = {}

    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
                "items": [],
            }

        band_summary[month]["total"] += 1
        band_summary[month]["items"].append({
            "milestone": a["milestone"],
            "answer": norm,
        })

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    for month in band_summary:
        total = band_summary[month]["total"]
        yes = band_summary[month]["yes"]
        band_summary[month]["yes_ratio"] = round(yes / total, 2) if total else 0.0

    return dict(sorted(band_summary.items()))


In [30]:
# ------------------------------------------------------------
# Q&A mode A — live parent
# ------------------------------------------------------------
def qna_agent_live(
    state: Dict[str, Any],
    category_key: str,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Run one domain interview in live mode using manual answers."""
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=3,
        max_bands=3,
        max_questions_total=9,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")
        for q in questions_by_month[month]:
            ans = input(q["question_text"] + "\n").strip()
            norm = normalize_answer(ans)
            asked.append({
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "raw_answer": ans,
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
                "answer_status": "ok",
            })

    dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(f"- [{item['months']}m] {item['milestone']} -> {item['norm_answer']}")

    band_summary = summarize_answers_by_band(asked)
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']}"
        )

    print(
        f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
        f"{dev_age} months (chronological age {chrono} months)"
    )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }


In [31]:
# ------------------------------------------------------------
# Q&A mode B — Gemini synthetic parent
# ------------------------------------------------------------
def gemini_parent_answer_one_question(
    child: Dict[str, Any],
    category_display: str,
    question_text: str,
    prior_answers: List[Dict[str, Any]],
    model: str = "gemini-2.5-flash",
    max_retries: int = 3,
) -> Dict[str, Any]:
    """Use Gemini to simulate one parent answer with retry/backoff protection."""
    if gemini_client is None:
        raise ValueError(
            "Gemini client is not available. Set GEMINI_API_KEY and rerun the environment setup cell."
        )

    if prior_answers:
        prior_text = "\n".join([f"- {x['question']} -> {x['answer']}" for x in prior_answers[-8:]])
    else:
        prior_text = "None yet."

    prompt = f"""
You are simulating a REAL parent answering developmental milestone questions about their child.

Rules:
- Answer ONLY from the child profile below.
- Be internally consistent across questions.
- Do NOT answer like a clinician or therapist.
- Answer like a parent who knows the child in daily life.
- Choose ONLY one answer:
  yes
  sometimes
  with_help
  no
  not_sure

Child profile:
- Name: {child.get('name')}
- Age: {child.get('chronological_months')} months
- Diagnosis / condition: {child.get('diagnosis')}
- Main concern: {child.get('concern')}

Current developmental category:
- {category_display}

Previous answers in this same interview:
{prior_text}

Question:
{question_text}

Return strict JSON only:
{{
  "answer": "yes|sometimes|with_help|no|not_sure",
  "reason": "one short parent-style reason"
}}
""".strip()

    last_error = None

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
            )

            raw_text = response.text if hasattr(response, "text") else str(response)
            parsed = extract_json_block(raw_text)

            answer = str(parsed.get("answer", "")).strip().lower().replace(" ", "_")
            if answer not in VALID_PARENT_ANSWERS:
                answer = "not_sure"

            return {
                "answer": answer,
                "reason": str(parsed.get("reason", "")).strip() or raw_text[:200],
                "raw_text": raw_text,
                "status": "ok",
            }

        except Exception as e:
            last_error = e
            print(f"Gemini warning on question: {question_text}")
            print(f"Attempt {attempt+1}/{max_retries} failed: {e}")

            if attempt < max_retries - 1:
                time.sleep(10 * (attempt + 1))

    return {
        "answer": "not_sure",
        "reason": f"Fallback after Gemini failure: {last_error}",
        "raw_text": "",
        "status": "api_error",
    }

def qna_agent_gemini_parent(
    state: Dict[str, Any],
    category_key: str,
    ask_limit_per_band: int = 3,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
    gemini_model: str = "gemini-2.5-flash",
    delay_between_questions_sec: int = 5,
) -> Dict[str, Any]:
    """Run one domain interview in synthetic-parent mode.

    API failures are marked as `api_error` and excluded from developmental-age scoring.
    """
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=3,
        max_bands=3,
        max_questions_total=9,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    prior_answers_for_gemini = []

    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")

        for q in questions_by_month[month][:ask_limit_per_band]:
            sim = gemini_parent_answer_one_question(
                child=state["child"],
                category_display=DOMAIN_CONFIG[category_key]["display"],
                question_text=q["question_text"],
                prior_answers=prior_answers_for_gemini,
                model=gemini_model,
                max_retries=3,
            )

            norm = normalize_answer(sim["answer"])

            asked_item = {
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "raw_answer": sim["answer"],
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
                "parent_reason": sim["reason"],
                "answer_status": sim.get("status", "ok"),
            }
            asked.append(asked_item)

            prior_answers_for_gemini.append({
                "question": q["question_text"],
                "answer": sim["answer"],
            })

            print(
                f"Q: {q['question_text']}\n"
                f"A (Gemini-parent): {sim['answer']} | "
                f"status: {sim.get('status', 'ok')} | "
                f"reason: {sim['reason']}"
            )

            time.sleep(delay_between_questions_sec)

    usable_answers = [a for a in asked if a.get("answer_status") != "api_error"]

    dev_age = compute_dev_age_from_answers(
        usable_answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(
            f"- [{item['months']}m] {item['milestone']} -> {item['norm_answer']} "
            f"| status: {item.get('answer_status', 'ok')} "
            f"| reason: {item.get('parent_reason', '')}"
        )

    band_summary = summarize_answers_by_band(usable_answers)
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']}"
        )

    print(
        f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
        f"{dev_age} months (chronological age {chrono} months)"
    )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }


In [32]:

# ------------------------------------------------------------
# Support tier + shared planning engine
# ------------------------------------------------------------
def get_support_tier(state: Dict[str, Any], category_key: str) -> str:
    """Assign an age-aware support tier using milestone gap and starting delay.

    We intentionally keep this deterministic rather than asking a model again.
    This makes it easier to inspect, tune, and explain to advisors.
    """
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    dev_age = state["dev_age"].get(category_key, chrono)
    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)

    gap = max(0, chrono - dev_age)

    # Age-aware thresholds:
    # smaller month gaps matter more in infancy than they do in older children.
    light_gap_threshold = max(2, round(0.10 * chrono))
    primary_gap_threshold = max(5, round(0.20 * chrono))

    light_delay_threshold = max(4, round(0.10 * chrono))
    primary_delay_threshold = max(6, round(0.20 * chrono))

    # Weighted support score:
    # milestone gap is more important than the rough starting delay estimate,
    # but both contribute to the tier decision.
    support_score = (
        0.65 * (gap / primary_gap_threshold) +
        0.35 * (delay_est / primary_delay_threshold)
    )

    if gap >= primary_gap_threshold or support_score >= 1.0:
        return "needs_special_support"

    if (
        (gap >= light_gap_threshold and delay_est >= light_delay_threshold)
        or support_score >= 0.60
    ):
        return "monitor_and_enrich"

    return "no_special_support"

def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    """Backward-compatible helper used elsewhere in the notebook."""
    return get_support_tier(state, category_key) == "no_special_support"

def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    """Select near-next-step milestones to drive the activity bank."""
    child = state["child"]
    dev_age = state["dev_age"].get(category_key, None)

    if dev_age is None:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key):
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    min_m = max(2, dev_age)
    max_m = min(60, dev_age + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)
    if subset.empty:
        return {"status": "success", "milestones": []}

    subset = subset.sort_values(["months", "milestone"]).copy()
    milestones = []
    seen = set()

    for _, row in subset.iterrows():
        key = (int(row["months"]), str(row["milestone"]).strip())
        if key in seen:
            continue
        seen.add(key)
        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
        })

    return {"status": "success", "milestones": milestones[:max_milestones]}

def get_category_activity_guardrails(category_key: str) -> str:
    """Return domain-specific activity guardrails to reduce cross-domain drift."""
    if category_key == "movement_and_physical":
        return """
Category-specific rules for Movement / Physical:
- Focus on posture, strength, balance, coordination, reaching, grasping, sitting, crawling, standing, walking, or fine-motor use.
- The main goal should be physical skill practice.
- Do NOT make this mainly about speech, naming, imitation of sounds, or social-emotional labeling.
"""

    if category_key == "language_and_communication":
        return """
Category-specific rules for Language / Communication:
- Focus on sounds, babbling, turn-taking vocalization, following simple verbal directions, gestures for communication, word use, imitation of sounds, naming, requesting, commenting, and comprehension.
- The main goal should be communication.
- Do NOT make this mainly about gross motor practice, reaching, balance, climbing, or general physical exercise.
- If an object is used, it should support communication, not be the main task.
"""

    if category_key == "social_and_emotional":
        return """
Category-specific rules for Social / Emotional:
- Focus on eye contact, shared attention, social reciprocity, imitation of facial expressions, joint play, turn-taking, response to name, emotional connection, and interaction with caregiver or peers.
- The main goal should be social engagement or emotional regulation.
- Do NOT make this mainly about motor strengthening or naming/labeling language tasks unless the social interaction is clearly primary.
"""

    if category_key == "cognitive":
        return """
Category-specific rules for Cognitive / Adaptive:
- Focus on attention, problem-solving, object permanence, simple cause-and-effect, routines, imitation of actions, functional play, early self-help, and adaptive understanding.
- The main goal should be cognitive/adaptive skill building.
- Do NOT make this mainly about speech production or gross motor strengthening unless the cognitive/adaptive task is clearly primary.
"""

    return ""

def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = "gpt-4o-mini",
    activities_per_category: int = 6,
) -> Dict[str, Any]:
    """Generate one activity bank per category.

    We generate activities category-by-category first, then schedule them deterministically.
    """
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]
    support_tier = get_support_tier(state, category_key)
    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["activity_banks"][category_key] = {
            "status": "no_special_support",
            "support_tier": support_tier,
            "summary": next_steps["message"],
            "activities": [],
        }
        return state["activity_banks"][category_key]

    dev_age = state["dev_age"][category_key]
    chrono_months = min(child["chronological_months"], 60)
    milestone_gap = max(0, chrono_months - dev_age)
    category_guardrails = get_category_activity_guardrails(category_key)

    milestone_lines = "\n".join([f"- ({m['months']} months) {m['milestone']}" for m in next_steps["milestones"]])
    if not milestone_lines:
        milestone_lines = "- No specific milestone items available in this range."

    if openai_client is None:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "goal": support_tier,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "goal": support_tier,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        state["activity_banks"][category_key] = {
            "status": "fallback",
            "support_tier": support_tier,
            "summary": f"Fallback activity bank used for {category_display}.",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

    prompt = f"""
You are a pediatric home-support planning agent helping a parent at home.

This is NOT a diagnosis and NOT a formal treatment plan.
Create a CATEGORY ACTIVITY BANK, not a day-by-day schedule.

Child:
- Name: {child['name']}
- Chronological age: {child['chronological_months']} months
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Support tier: {support_tier}
- Estimated developmental age in this category: {dev_age} months
- Estimated milestone gap in this category: {milestone_gap} months

Relevant milestone targets:
{milestone_lines}

Category-specific guardrails:
{category_guardrails}

Task:
Create {activities_per_category} realistic home activities for this category.

Instructions:
1. Activities must fit the child's chronological age and estimated developmental level.
2. Activities should be practical for home.
3. Keep language parent-friendly.
4. Include a mix of current-level practice and near next-step practice.
5. Most activities should be short and realistic for home: usually 3, 5, 7, or 10 minutes.
6. Avoid making many 15-minute activities unless clearly justified.
7. Rare exception: some activities may benefit from longer continuous time, such as a short playdate or peer practice activity.
8. Only mark an activity as extended if it is clearly justified.
9. Extended activities should usually be 15 or 20 minutes.
10. Do not make more than 1 or 2 extended activities in this category.
11. Each activity must clearly belong to THIS category first and foremost.
12. Avoid cross-domain drift. The main goal of the activity should be obvious from the category.
13. If an activity could fit multiple domains, choose the version that best matches this category's core skill.
14. For each activity, set "goal" to one of: "needs_special_support", "monitor_and_enrich", or "current_or_next", whichever fits best.
15. Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "activities": [
    {{
      "activity_id": "1",
      "title": "...",
      "instructions": "...",
      "duration_min": 5,
      "materials": "...",
      "level": "current_or_next",
      "category": "<category name>",
      "goal": "{support_tier}",
      "is_extended_activity": false,
      "extended_reason": ""
    }}
  ]
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "You return strict JSON only and stay non-diagnostic, practical, and parent-friendly."
                },
                {"role": "user", "content": prompt},
            ],
        )

        bank = json.loads(resp.choices[0].message.content)

        if "activities" not in bank or not isinstance(bank["activities"], list):
            bank["activities"] = []

        for idx, activity in enumerate(bank["activities"], start=1):
            activity.setdefault("activity_id", f"{category_key}_{idx}")
            activity.setdefault("category", category_display)
            activity.setdefault("duration_min", 5)
            activity.setdefault("materials", "common household items")
            activity.setdefault("level", "current_or_next")
            activity["goal"] = support_tier
            activity.setdefault("is_extended_activity", False)
            activity.setdefault("extended_reason", "")

        state["activity_banks"][category_key] = {
            "status": "success",
            "support_tier": support_tier,
            "summary": bank.get("summary", f"Created activity bank for {category_display}."),
            "activities": bank["activities"][:activities_per_category],
        }
        return state["activity_banks"][category_key]

    except Exception as e:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "goal": support_tier,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "goal": support_tier,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        state["activity_banks"][category_key] = {
            "status": "fallback",
            "support_tier": support_tier,
            "summary": f"Fallback activity bank used for {category_display} because OpenAI failed: {e}",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

def allocate_weekly_slots(state: Dict[str, Any]) -> Dict[str, Any]:
    """Allocate weekly minutes across categories using support tier + milestone gap."""
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * 5

    supported_categories = []
    gap_by_category = {}
    weight_by_category = {}
    base_minutes_by_category = {}

    for category_key in DOMAIN_CONFIG:
        tier = get_support_tier(state, category_key)
        if tier == "no_special_support":
            continue

        supported_categories.append(category_key)
        dev_age = state["dev_age"].get(category_key, chrono)
        gap = max(0, chrono - dev_age)
        gap_by_category[category_key] = gap

        if tier == "needs_special_support":
            weight_by_category[category_key] = max(1.0, float(gap))
            base_minutes_by_category[category_key] = 5
        else:  # monitor_and_enrich
            weight_by_category[category_key] = max(1.0, float(gap)) * 0.5
            base_minutes_by_category[category_key] = 3

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "gap_by_category": {},
            "target_minutes_by_category": {},
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    total_base = sum(base_minutes_by_category.values())

    if total_base > weekly_minutes:
        scale = weekly_minutes / total_base
        base_minutes_by_category = {
            k: max(1, int(round(v * scale)))
            for k, v in base_minutes_by_category.items()
        }

    target_minutes_by_category = base_minutes_by_category.copy()
    used_minutes = sum(target_minutes_by_category.values())
    remaining_minutes = max(0, weekly_minutes - used_minutes)

    weights = weight_by_category.copy()
    total_weight = sum(weights.values())

    if total_weight > 0 and remaining_minutes > 0:
        for k in supported_categories:
            extra = round(remaining_minutes * (weights[k] / total_weight))
            target_minutes_by_category[k] += extra

    total_target = sum(target_minutes_by_category.values())
    while total_target > weekly_minutes:
        biggest = max(target_minutes_by_category, key=target_minutes_by_category.get)
        if target_minutes_by_category[biggest] > 1:
            target_minutes_by_category[biggest] -= 1
            total_target -= 1
        else:
            break

    allocation = {
        "daily_time_min": daily_time_min,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "gap_by_category": gap_by_category,
        "target_minutes_by_category": target_minutes_by_category,
    }
    state["weekly_slot_allocation"] = allocation
    return allocation

def pick_activity_that_fits(
    activities: List[Dict[str, Any]],
    used_indices: set,
    remaining_minutes: int,
) -> Optional[Dict[str, Any]]:
    """Greedy helper: pick the shortest unused activity that fits the remaining minutes."""
    candidates = []
    for idx, activity in enumerate(activities):
        if idx in used_indices:
            continue
        duration = int(activity.get("duration_min", 5))
        if duration <= remaining_minutes:
            candidates.append((duration, idx, activity))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: x[0])
    duration, idx, activity = candidates[0]
    used_indices.add(idx)
    return activity

def _recompute_assigned_minutes(days: Dict[str, Any], target_minutes_by_category: Dict[str, int]) -> Dict[str, int]:
    """Recompute assigned minutes by category from the current day schedule."""
    assigned_minutes = {k: 0 for k in target_minutes_by_category.keys()}
    for _, day_info in days.items():
        for item in day_info.get("items", []):
            k = item.get("category_key")
            if k in assigned_minutes:
                assigned_minutes[k] += int(item.get("duration_min", 0))
    return assigned_minutes

def ensure_minimum_presence_for_monitor_categories(state: Dict[str, Any], days: Dict[str, Any]) -> Dict[str, Any]:
    """Ensure monitor/enrich categories get at least one short activity when possible.

    First, try to place a short activity into a day with free time.
    If every day is too full, try a gentle swap:
    replace one activity from a category that is currently OVER its target
    so the missing monitor/enrich category is not invisible in the weekly plan.
    """
    allocation = state.get("weekly_slot_allocation", {})
    target_minutes_by_category = allocation.get("target_minutes_by_category", {})
    child = state["child"]
    daily_time_min = int(child["daily_time_min"])

    assigned_minutes = _recompute_assigned_minutes(days, target_minutes_by_category)

    for category_key in target_minutes_by_category.keys():
        tier = get_support_tier(state, category_key)

        if tier != "monitor_and_enrich":
            continue
        if target_minutes_by_category.get(category_key, 0) <= 0:
            continue
        if assigned_minutes.get(category_key, 0) > 0:
            continue

        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])
        if not activities:
            continue

        short_candidates = [a for a in activities if int(a.get("duration_min", 5)) <= 5]
        if short_candidates:
            chosen = min(short_candidates, key=lambda a: int(a.get("duration_min", 5)))
        else:
            chosen = min(activities, key=lambda a: int(a.get("duration_min", 5)))

        chosen_duration = int(chosen.get("duration_min", 5))
        chosen_item = {
            "category_key": category_key,
            "category": DOMAIN_CONFIG[category_key]["display"],
            "title": chosen.get("title"),
            "instructions": chosen.get("instructions"),
            "duration_min": chosen_duration,
            "materials": chosen.get("materials", "common household items"),
            "level": chosen.get("level", "current_or_next"),
            "goal": chosen.get("goal", "monitor_and_enrich"),
        }

        # Pass 1: direct placement into a day that still has enough room.
        day_names_by_room = sorted(
            days.keys(),
            key=lambda d: daily_time_min - int(days[d].get("total_minutes", 0)),
            reverse=True,
        )

        placed = False
        for day_name in day_names_by_room:
            day_info = days[day_name]
            remaining = daily_time_min - int(day_info.get("total_minutes", 0))
            if remaining >= chosen_duration:
                day_info["items"].append(chosen_item)
                day_info["total_minutes"] += chosen_duration
                assigned_minutes = _recompute_assigned_minutes(days, target_minutes_by_category)
                placed = True
                break

        if placed:
            continue

        # Pass 2: gentle swap with a category that is over target.
        donor_candidates = []
        for day_name, day_info in days.items():
            for item_idx, item in enumerate(day_info.get("items", [])):
                donor_category = item.get("category_key")
                if donor_category not in assigned_minutes or donor_category == category_key:
                    continue

                donor_duration = int(item.get("duration_min", 0))
                donor_surplus = assigned_minutes.get(donor_category, 0) - target_minutes_by_category.get(donor_category, 0)
                new_total = int(day_info.get("total_minutes", 0)) - donor_duration + chosen_duration

                if donor_surplus > 0 and new_total <= daily_time_min:
                    donor_candidates.append((
                        donor_surplus,
                        donor_duration,
                        day_name,
                        item_idx,
                        donor_category,
                    ))

        if donor_candidates:
            donor_candidates.sort(
                key=lambda x: (
                    x[0],                    # largest surplus first
                    x[1] >= chosen_duration, # prefer donor durations that fully cover the new item
                    x[1],                    # then larger donor duration
                ),
                reverse=True,
            )

            _, _, day_name, item_idx, donor_category = donor_candidates[0]
            day_info = days[day_name]
            removed_item = day_info["items"].pop(item_idx)
            day_info["total_minutes"] -= int(removed_item.get("duration_min", 0))
            day_info["items"].append(chosen_item)
            day_info["total_minutes"] += chosen_duration
            assigned_minutes = _recompute_assigned_minutes(days, target_minutes_by_category)

    return days

def pick_weekly_bonus_extended_activity(
    state: Dict[str, Any],
    extended_in_schedule_threshold: int = 15,
    bonus_extended_cap_min: int = 20,
) -> Optional[Dict[str, Any]]:
    """Optional extended activity added only as a weekly bonus, not counted in the daily budget."""
    daily_time_min = int(state["child"]["daily_time_min"])

    if daily_time_min >= extended_in_schedule_threshold:
        return None

    allocation = state.get("weekly_slot_allocation", {})
    gap_by_category = allocation.get("gap_by_category", {})

    candidate_categories = sorted(
        gap_by_category.keys(),
        key=lambda k: gap_by_category[k],
        reverse=True,
    )

    for category_key in candidate_categories:
        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])

        for activity in activities:
            if not activity.get("is_extended_activity", False):
                continue

            duration = int(activity.get("duration_min", 5))
            if duration > bonus_extended_cap_min:
                continue

            return {
                "category_key": category_key,
                "category": DOMAIN_CONFIG[category_key]["display"],
                "title": activity.get("title"),
                "instructions": activity.get("instructions"),
                "duration_min": duration,
                "materials": activity.get("materials", "common household items"),
                "level": activity.get("level", "current_or_next"),
                "goal": activity.get("goal", ""),
                "extended_reason": activity.get("extended_reason", ""),
            }

    return None

def build_weekly_schedule(state: Dict[str, Any]) -> Dict[str, Any]:
    """Build the final weekly plan from activity banks and allocated minutes."""
    if "weekly_slot_allocation" not in state:
        allocate_weekly_slots(state)

    allocation = state["weekly_slot_allocation"]
    daily_time_min = allocation["daily_time_min"]
    target_minutes_by_category = allocation["target_minutes_by_category"]

    days = {
        "day_1": {"items": [], "total_minutes": 0},
        "day_2": {"items": [], "total_minutes": 0},
        "day_3": {"items": [], "total_minutes": 0},
        "day_4": {"items": [], "total_minutes": 0},
        "day_5": {"items": [], "total_minutes": 0},
    }

    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "days": days,
        }
        return state["weekly_schedule"]

    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    used_activity_indices = {k: set() for k in target_minutes_by_category.keys()}
    day_names = list(days.keys())

    progress_made = True
    while progress_made:
        progress_made = False

        categories_in_priority_order = sorted(
            target_minutes_by_category.keys(),
            key=lambda k: target_minutes_by_category[k] - assigned_minutes_by_category[k],
            reverse=True,
        )

        for day_name in day_names:
            remaining_day_minutes = daily_time_min - days[day_name]["total_minutes"]
            if remaining_day_minutes <= 0:
                continue

            for category_key in categories_in_priority_order:
                remaining_cat_minutes = (
                    target_minutes_by_category[category_key] - assigned_minutes_by_category[category_key]
                )
                if remaining_cat_minutes <= 0:
                    continue

                bank = state["activity_banks"].get(category_key, {})
                activities = bank.get("activities", [])
                if not activities:
                    continue

                activity = pick_activity_that_fits(
                    activities=activities,
                    used_indices=used_activity_indices[category_key],
                    remaining_minutes=remaining_day_minutes,
                )
                if activity is None:
                    continue

                duration = int(activity.get("duration_min", 5))
                days[day_name]["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": activity.get("title"),
                    "instructions": activity.get("instructions"),
                    "duration_min": duration,
                    "materials": activity.get("materials", "common household items"),
                    "level": activity.get("level", "current_or_next"),
                    "goal": activity.get("goal", ""),
                })

                days[day_name]["total_minutes"] += duration
                assigned_minutes_by_category[category_key] += duration
                progress_made = True
                break

    # Small repair pass: if a monitor/enrich category got squeezed out completely,
    # try to place one short activity into any day that still has room.
    days = ensure_minimum_presence_for_monitor_categories(state, days)

    # Recompute assigned minutes after the repair pass.
    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    for _, day_info in days.items():
        for item in day_info.get("items", []):
            k = item.get("category_key")
            if k in assigned_minutes_by_category:
                assigned_minutes_by_category[k] += int(item.get("duration_min", 0))

    bonus_activity = pick_weekly_bonus_extended_activity(
        state,
        extended_in_schedule_threshold=15,
        bonus_extended_cap_min=20,
    )

    schedule = {
        "status": "success",
        "summary": "Weekly schedule built from category activity banks and true daily minute limits.",
        "daily_time_min": daily_time_min,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "days": days,
        "weekly_bonus_activity": bonus_activity,
    }

    state["weekly_schedule"] = schedule
    return schedule

def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    """Summarize final case outputs into a tidy dataframe."""
    child = state["child"]
    rows = []
    allocation = state.get("weekly_slot_allocation", {}).get("target_minutes_by_category", {})

    for category_key in DOMAIN_CONFIG:
        delay_info = state["delay_estimates"].get(category_key, {})
        dev_age = state["dev_age"].get(category_key)
        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if dev_age is None else max(0, chrono_for_gap - dev_age)
        bank = state.get("activity_banks", {}).get(category_key, {})

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "estimated_dev_age_months": dev_age,
            "milestone_gap_months": milestone_gap,
            "support_tier": get_support_tier(state, category_key),
            "needs_special_support": not no_special_support_needed(state, category_key),
            "weekly_target_minutes_for_category": allocation.get(category_key, 0),
            "activity_bank_status": bank.get("status", "missing"),
            "activity_bank_summary": bank.get("summary", ""),
        })

    return pd.DataFrame(rows)

def display_weekly_schedule(state: Dict[str, Any]) -> None:
    """Print the final weekly schedule in a readable form."""
    schedule = state.get("weekly_schedule", {})

    print_banner("WEEKLY SCHEDULE")

    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return

    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Target minutes by category:", schedule.get("target_minutes_by_category", {}))
    print("Assigned minutes by category:", schedule.get("assigned_minutes_by_category", {}))

    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)

        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue

        for item in items:
            print(f"- [{item['category']}] {item['title']} ({item['duration_min']} min)")
            print(f"  Goal: {item.get('goal', '')}")
            print(f"  Instructions: {item['instructions']}")
            print(f"  Materials: {item['materials']}")

    bonus = schedule.get("weekly_bonus_activity")
    if bonus:
        print("\n" + "=" * 100)
        print("OPTIONAL WEEKLY BONUS ACTIVITY")
        print("=" * 100)
        print(f"- [{bonus['category']}] {bonus['title']} ({bonus['duration_min']} min)")
        print(f"  Goal: {bonus.get('goal', '')}")
        print(f"  Instructions: {bonus['instructions']}")
        print(f"  Materials: {bonus['materials']}")
        print(f"  Why extended: {bonus.get('extended_reason', '')}")

def run_downstream_planning(state: Dict[str, Any], display_schedule: bool = True) -> pd.DataFrame:
    """Run the shared post-interview planning pipeline."""
    for category_key in DOMAIN_CONFIG:
        generate_category_activity_bank(state, category_key)

    allocate_weekly_slots(state)
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state)

    if display_schedule:
        display_weekly_schedule(state)

    return summary_df


In [33]:
# ------------------------------------------------------------
# Main runners
# ------------------------------------------------------------
def run_all_categories_live():
    """Run one full case in live-answer mode."""
    state = new_state()
    intake_agent_live(state)
    print_child_reference(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def run_all_categories_gemini_parent(
    case: Dict[str, Any],
    daily_time_min: int = 10,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run one full case in synthetic-parent mode using Gemini."""
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_child_reference(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=3,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
            gemini_model=gemini_model,
            delay_between_questions_sec=5,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def print_case_reference(case: dict, daily_time_min: int):
    print("\n" + "=" * 110)
    print(f"STARTING {case['case_id']} | {case['child_name']} | {case['age_label']} | {case['diagnosis']}")
    print("=" * 110)
    print(f"Concerns: {case['concerns']}")
    print(f"Daily time: {daily_time_min} min/day")

def run_case_live_from_prefilled_case(
    case: dict,
    default_daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
):
    """Run a prefilled case live while still asking milestone questions manually."""
    raw_daily = input(
        f"[{case['case_id']}] Daily minutes available? Press Enter for default {default_daily_time_min}: "
    ).strip()
    daily_time_min = int(raw_daily) if raw_daily else int(default_daily_time_min)

    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def run_case_gemini_from_prefilled_case(
    case: dict,
    daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run a prefilled case fully in synthetic-parent mode."""
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=3,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
            gemini_model=gemini_model,
            delay_between_questions_sec=5,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df


In [34]:
# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------
cases = [
    {"case_id": "C01", "child_name": "Noah", "age_months": 10, "age_label": "10 months",
     "diagnosis": "Down syndrome",
     "concerns": "Not sitting independently yet, Hypotonia, slow feeding, socially engaged"},

    {"case_id": "C02", "child_name": "Maya", "age_months": 18, "age_label": "18 months",
     "diagnosis": "No formal diagnosis",
     "concerns": "No words yet, uses gestures, understands simple commands, good eye contact"},

    {"case_id": "C03", "child_name": "Leo", "age_months": 26, "age_label": "2y 2m",
     "diagnosis": "Autism spectrum disorder",
     "concerns": "Limited eye contact and few words, Repetitive play, transitions hard, picky eating"},

    {"case_id": "C04", "child_name": "Sofia", "age_months": 37, "age_label": "3y 1m",
     "diagnosis": "Prematurity history",
     "concerns": "Speech hard to understand, Fine motor delay, good comprehension"},

    {"case_id": "C05", "child_name": "Ethan", "age_months": 54, "age_label": "4y 6m",
     "diagnosis": "Cerebral palsy",
     "concerns": "Trouble keeping up in preschool, Self-care delays, frequent falls, difficulty with stairs, uses walker sometimes, good cognition"},

    {"case_id": "C06", "child_name": "Amina", "age_months": 60, "age_label": "5 years",
     "diagnosis": "Global developmental delay",
     "concerns": "Learning and routines are hard, Short attention span, follows only simple directions"},

    {"case_id": "C07", "child_name": "Lucas", "age_months": 14, "age_label": "14 months",
     "diagnosis": "Fragile X syndrome",
     "concerns": "Not walking, limited babbling, Sensory sensitivity, shy socially"},

    {"case_id": "C08", "child_name": "Emma", "age_months": 32, "age_label": "2y 8m",
     "diagnosis": "Suspected childhood apraxia of speech",
     "concerns": "Very limited speech output, Uses gestures, gets frustrated, inconsistent sounds"},

    {"case_id": "C09", "child_name": "Zain", "age_months": 48, "age_label": "4 years",
     "diagnosis": "Attention and self-regulation concerns (possible ADHD)",
     "concerns": "Very active, impulsive, Big emotional reactions, interrupts, difficulty following routines, social interest intact"},

    {"case_id": "C10", "child_name": "Isla", "age_months": 44, "age_label": "3y 8m",
     "diagnosis": "SYNGAP1-related disorder",
     "concerns": "Delayed speech and balance, Attention issues, clumsy gait, limited pretend play"},

    {"case_id": "C11", "child_name": "Aria", "age_months": 20, "age_label": "20 months",
     "diagnosis": "No diagnosis",
     "concerns": "Only ~10 words, very active, good eye contact, understands well"},

    {"case_id": "C12", "child_name": "Mateo", "age_months": 30, "age_label": "2y 6m",
     "diagnosis": "No formal diagnosis",
     "concerns": "Good eye contact but no words, lines up toys, responds to name inconsistently"},

    {"case_id": "C13", "child_name": "Luna", "age_months": 42, "age_label": "3y 6m",
     "diagnosis": "Speech delay",
     "concerns": "Limited speech, understands well, parents have very limited time for activities"},

    {"case_id": "C14", "child_name": "Ryan", "age_months": 50, "age_label": "4y 2m",
     "diagnosis": "No diagnosis",
     "concerns": "Tantrums, difficulty with transitions, strong language skills, social but rigid"},
]

df_cases = pd.DataFrame(cases)
display(df_cases)


,case_id,child_name,age_months,age_label,diagnosis,concerns
0,C01,Noah,10,10 months,Down syndrome,"Not sitting independently yet, Hypotonia, slow..."
1,C02,Maya,18,18 months,No formal diagnosis,"No words yet, uses gestures, understands simpl..."
2,C03,Leo,26,2y 2m,Autism spectrum disorder,"Limited eye contact and few words, Repetitive ..."
3,C04,Sofia,37,3y 1m,Prematurity history,"Speech hard to understand, Fine motor delay, g..."
4,C05,Ethan,54,4y 6m,Cerebral palsy,"Trouble keeping up in preschool, Self-care del..."
5,C06,Amina,60,5 years,Global developmental delay,"Learning and routines are hard, Short attentio..."
6,C07,Lucas,14,14 months,Fragile X syndrome,"Not walking, limited babbling, Sensory sensiti..."
7,C08,Emma,32,2y 8m,Suspected childhood apraxia of speech,"Very limited speech output, Uses gestures, get..."
8,C09,Zain,48,4 years,Attention and self-regulation concerns (possib...,"Very active, impulsive, Big emotional reaction..."
9,C10,Isla,44,3y 8m,SYNGAP1-related disorder,"Delayed speech and balance, Attention issues, ..."


In [35]:
# ------------------------------------------------------------
# Advisor-report helpers
# ------------------------------------------------------------
def severity_from_gap(gap_months: float) -> str:
    gap_months = 0 if pd.isna(gap_months) else float(gap_months)
    if gap_months <= 6:
        return "minimal / mild"
    elif gap_months <= 18:
        return "moderate"
    else:
        return "significant"

def split_concerns_to_bullets(concern_text: str, max_items: int = 4):
    items = [x.strip(" -•\n\t") for x in re.split(r"[,\n;]+", str(concern_text)) if x.strip()]
    return items[:max_items]

def extract_parent_qa_lines(state: dict, max_lines_for_pdf: int = 12):
    full_lines = []
    for category_key in DOMAIN_CONFIG.keys():
        answers = state.get("qna", {}).get(category_key, [])
        for a in answers:
            milestone = str(a.get("milestone", "")).strip()
            norm_answer = str(a.get("norm_answer", a.get("raw_answer", ""))).strip()
            months = a.get("months", "")
            status = a.get("answer_status", "ok")
            full_lines.append(f"[{months}m] {milestone} -> {norm_answer} ({status})")

    pdf_lines = full_lines[:max_lines_for_pdf]
    return full_lines, pdf_lines

def extract_focus_areas(summary_df: pd.DataFrame, max_items: int = 3):
    if summary_df is None or summary_df.empty:
        return []

    work = summary_df.copy()
    work = work[work["support_tier"].isin(["needs_special_support", "monitor_and_enrich"])].copy()
    if work.empty:
        return []

    work["tier_rank"] = work["support_tier"].map({
        "needs_special_support": 2,
        "monitor_and_enrich": 1,
        "no_special_support": 0,
    })
    sort_col = "milestone_gap_months" if "milestone_gap_months" in work.columns else "estimated_delay_months"
    work = work.sort_values(["tier_rank", sort_col], ascending=[False, False])

    return work["category"].tolist()[:max_items]

def extract_activities_for_case(state: dict, max_activities: int = 5):
    activities = []
    weekly_schedule = state.get("weekly_schedule", {})
    days = weekly_schedule.get("days", {})

    if isinstance(days, dict):
        for day_label, day_info in days.items():
            day_items = day_info.get("items", [])
            for item in day_items:
                activities.append({
                    "day": day_label,
                    "category": item.get("category", item.get("category_display", "")),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", item.get("category", ""))),
                })

    if not activities:
        for category_key, bank in state.get("activity_banks", {}).items():
            for item in bank.get("activities", [])[:2]:
                activities.append({
                    "day": "",
                    "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", "current_or_next")),
                })

    return activities[:max_activities]

def build_case_payload(case: dict, state: dict, summary_df: pd.DataFrame):
    concerns_bullets = split_concerns_to_bullets(case.get("concerns", ""), max_items=4)
    parent_qa_full, parent_qa_pdf = extract_parent_qa_lines(state, max_lines_for_pdf=12)
    focus_areas = extract_focus_areas(summary_df, max_items=3)
    activities = extract_activities_for_case(state, max_activities=5)

    domain_rows = []
    if summary_df is not None and not summary_df.empty:
        for _, row in summary_df.iterrows():
            gap = row.get("milestone_gap_months", row.get("estimated_delay_months", None))
            domain_rows.append({
                "category": row.get("category", ""),
                "estimated_dev_age_months": row.get("estimated_dev_age_months", ""),
                "gap_months": gap,
                "support_tier": row.get("support_tier", ""),
                "severity": severity_from_gap(gap if gap is not None else 0),
                "needs_special_support": row.get("needs_special_support", True),
                "summary": row.get("activity_bank_summary", ""),
            })

    support_categories = [r["category"] for r in domain_rows if r.get("support_tier") in {"needs_special_support", "monitor_and_enrich"}]
    if support_categories:
        summary_text = (
            f"This plan prioritizes {', '.join(support_categories[:3])} for {case['child_name']}. "
            f"The activities are intended to be short, parent-friendly, and aligned with the developmental areas that appeared most delayed or most worth enriching in the interview."
        )
    else:
        summary_text = (
            f"This output did not identify a strong need for special support across the reviewed categories, "
            f"but the family can continue monitoring development and bring remaining questions to the clinician."
        )

    payload = {
        "case_id": case.get("case_id"),
        "child_name": case.get("child_name"),
        "age_label": case.get("age_label"),
        "age_months": case.get("age_months"),
        "diagnosis": case.get("diagnosis"),
        "concerns_bullets": concerns_bullets,
        "daily_time_min": state.get("child", {}).get("daily_time_min", ""),
        "parent_qa_full": parent_qa_full,
        "parent_qa_pdf": parent_qa_pdf,
        "domain_rows": domain_rows,
        "focus_areas": focus_areas,
        "activities": activities,
        "summary_text": summary_text,
    }
    return payload

def print_case_screen_summary(payload: dict):
    print("\n" + "=" * 110)
    print(f"CASE {payload['case_id']} | {payload['child_name']} | {payload['age_label']} | {payload['diagnosis']}")
    print("=" * 110)

    print("\n1. CHILD PROFILE")
    print(f"- Name: {payload['child_name']}")
    print(f"- Age: {payload['age_label']} ({payload['age_months']} months)")
    print(f"- Diagnosis: {payload['diagnosis']}")
    print(f"- Daily time available: {payload['daily_time_min']} min/day")
    print("- Key concerns:")
    for c in payload["concerns_bullets"]:
        print(f"  • {c}")

    print("\n2. PARENT INPUT (full transcript)")
    for line in payload["parent_qa_full"]:
        print(f"- {line}")

    print("\n3. GENEX OUTPUT")
    domain_df = pd.DataFrame(payload["domain_rows"])
    if not domain_df.empty:
        display(domain_df[["category", "estimated_dev_age_months", "gap_months", "support_tier", "severity"]])

    print("\nFocus areas:")
    for x in payload["focus_areas"]:
        print(f"  • {x}")

    print("\nDaily activities:")
    for i, a in enumerate(payload["activities"], start=1):
        print(f"{i}. {a['title']} | {a['category']} | {a['duration_min']} min")
        print(f"   Goal: {a['goal']}")
        print(f"   Description: {a['description']}")

    print("\n4. SUMMARY")
    print(payload["summary_text"])

    print("\n5. ADVISOR REVIEW SECTION")
    print("Rate 1–5:")
    print("- Clinical appropriateness")
    print("- Safety")
    print("- Practicality for parents")
    print("- Clarity")
    print("- Overall usefulness")
    print("Short feedback:")
    print("- What would you change?")
    print("- What is missing?")
    print("- Any concerns?")


In [36]:
# ------------------------------------------------------------
# PDF rendering + rating sheet
# ------------------------------------------------------------
def wrap_text_to_width(text, font_name, font_size, max_width):
    words = str(text).split()
    if not words:
        return [""]

    lines = []
    current = words[0]
    for word in words[1:]:
        trial = current + " " + word
        if stringWidth(trial, font_name, font_size) <= max_width:
            current = trial
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines

def draw_wrapped_text(c, text, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=None):
    lines = wrap_text_to_width(text, font_name, font_size, max_width)
    original_lines = lines[:]
    if max_lines is not None:
        lines = lines[:max_lines]
        if len(original_lines) > max_lines and lines:
            lines[-1] = lines[-1][:max(0, len(lines[-1]) - 3)] + "..."
    c.setFont(font_name, font_size)
    for line in lines:
        c.drawString(x, y, line)
        y -= line_height
    return y

def draw_bullets(c, items, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_items=None):
    items = list(items)
    if max_items is not None:
        items = items[:max_items]

    for item in items:
        bullet_lines = wrap_text_to_width(str(item), font_name, font_size, max_width - 12)
        if bullet_lines:
            c.drawString(x, y, u"\u2022")
            c.drawString(x + 10, y, bullet_lines[0])
            y -= line_height
            for extra in bullet_lines[1:]:
                c.drawString(x + 10, y, extra)
                y -= line_height
        else:
            y -= line_height
    return y

def draw_box(c, x, y_top, w, h, title):
    c.roundRect(x, y_top - h, w, h, 8, stroke=1, fill=0)
    c.setFont("Helvetica-Bold", 10)
    c.drawString(x + 8, y_top - 14, title)

def render_case_page(c, payload: dict):
    page_w, page_h = landscape(LETTER)
    margin = 0.35 * inch

    left_x = margin
    mid_x = page_w / 2 + 0.10 * inch
    col_w = (page_w / 2) - (1.5 * margin)

    c.setFont("Helvetica-Bold", 15)
    c.drawString(margin, page_h - 0.45 * inch, f"{payload['case_id']} — {payload['child_name']} ({payload['age_label']})")
    c.setFont("Helvetica", 10)
    c.drawString(margin, page_h - 0.68 * inch, f"Diagnosis: {payload['diagnosis']}")

    top_y = page_h - 0.90 * inch
    profile_h = 1.60 * inch
    input_h = 2.55 * inch
    output_h = 2.20 * inch
    activity_h = 2.10 * inch
    summary_h = 0.85 * inch
    rating_h = 1.20 * inch

    draw_box(c, left_x, top_y, col_w, profile_h, "1. Child Profile")
    draw_box(c, left_x, top_y - profile_h - 0.10 * inch, col_w, input_h, "2. Parent Input (questions + answers)")
    draw_box(c, mid_x, top_y, col_w, output_h, "3. Genex Output")
    draw_box(c, mid_x, top_y - output_h - 0.10 * inch, col_w, activity_h, "4. Daily Activities")

    bottom_top = page_h - 6.65 * inch
    full_w = page_w - 2 * margin
    draw_box(c, margin, bottom_top, full_w, summary_h, "5. Summary")
    draw_box(c, margin, bottom_top - summary_h - 0.10 * inch, full_w, rating_h, "6. Advisor Review")

    y = top_y - 28
    c.setFont("Helvetica", 9)
    c.drawString(left_x + 10, y, f"Name: {payload['child_name']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Age: {payload['age_label']} ({payload['age_months']} months)")
    y -= 12
    c.drawString(left_x + 10, y, f"Diagnosis: {payload['diagnosis']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Daily time: {payload['daily_time_min']} min/day")
    y -= 16
    c.setFont("Helvetica-Bold", 9)
    c.drawString(left_x + 10, y, "Key concerns:")
    y -= 11
    y = draw_bullets(c, payload["concerns_bullets"], left_x + 12, y, col_w - 20, max_items=4)

    y = top_y - profile_h - 0.10 * inch - 28
    y = draw_bullets(c, payload["parent_qa_pdf"], left_x + 12, y, col_w - 20, font_size=7.8, line_height=9, max_items=12)
    c.setFont("Helvetica-Oblique", 7.5)
    c.drawString(left_x + 10, top_y - profile_h - input_h - 0.10 * inch + 10, "Full transcript is preserved in JSON output.")

    y = top_y - 28
    domain_rows = payload["domain_rows"][:4]
    c.setFont("Helvetica-Bold", 8.5)
    c.drawString(mid_x + 10, y, "Domain")
    c.drawString(mid_x + 150, y, "Dev age")
    c.drawString(mid_x + 220, y, "Gap")
    c.drawString(mid_x + 270, y, "Tier")
    y -= 11
    c.setFont("Helvetica", 8.2)
    for row in domain_rows:
        c.drawString(mid_x + 10, y, str(row["category"])[:24])
        c.drawString(mid_x + 150, y, str(row["estimated_dev_age_months"]))
        c.drawString(mid_x + 220, y, str(row["gap_months"]))
        c.drawString(mid_x + 270, y, str(row["support_tier"])[:18])
        y -= 10

    y -= 8
    c.setFont("Helvetica-Bold", 9)
    c.drawString(mid_x + 10, y, "Focus areas:")
    y -= 11
    y = draw_bullets(c, payload["focus_areas"], mid_x + 12, y, col_w - 20, max_items=3)

    y = top_y - output_h - 0.10 * inch - 28
    for i, a in enumerate(payload["activities"][:5], start=1):
        line = f"{i}. {a['title']} ({a['duration_min']} min) — {a['goal']}"
        y = draw_wrapped_text(c, line, mid_x + 10, y, col_w - 20, font_name="Helvetica-Bold", font_size=8.2, line_height=9, max_lines=2)
        y = draw_wrapped_text(c, a["description"], mid_x + 16, y, col_w - 26, font_name="Helvetica", font_size=7.8, line_height=8.5, max_lines=2)
        y -= 4

    y = bottom_top - 28
    y = draw_wrapped_text(c, payload["summary_text"], margin + 10, y, full_w - 20, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=4)

    y = bottom_top - summary_h - 0.10 * inch - 28
    rating_lines = [
        "Rate 1–5: Clinical appropriateness | Safety | Practicality for parents | Clarity | Overall usefulness",
        "Short feedback: What would you change? | What is missing? | Any concerns?"
    ]
    for line in rating_lines:
        y = draw_wrapped_text(c, line, margin + 10, y, full_w - 20, font_name="Helvetica", font_size=9, line_height=11, max_lines=2)

    c.showPage()

def build_advisor_rating_sheet(df_cases: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df_cases.iterrows():
        rows.append({
            "case_id": row["case_id"],
            "child_name": row["child_name"],
            "age_months": row["age_months"],
            "diagnosis": row["diagnosis"],
            "clinical_appropriateness_1to5": "",
            "safety_1to5": "",
            "practicality_for_parents_1to5": "",
            "clarity_1to5": "",
            "overall_usefulness_1to5": "",
            "what_would_you_change": "",
            "what_is_missing": "",
            "any_concerns": "",
        })
    return pd.DataFrame(rows)


In [37]:
# ------------------------------------------------------------
# Generic advisor-report export runner
# ------------------------------------------------------------
def run_cases_and_export(
    df_cases: pd.DataFrame,
    mode: str = "live",   # "live" or "gemini"
    output_dir: str = "outputs/advisor_case_pack_v4",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = True,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run all cases and export the advisor packet files."""
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))
    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)

    for i, (_, row) in enumerate(df_cases.iterrows(), start=1):
        case = row.to_dict()

        if mode == "live":
            state, summary_df = run_case_live_from_prefilled_case(
                case=case,
                default_daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.60,
            )
        else:
            state, summary_df = run_case_gemini_from_prefilled_case(
                case=case,
                daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.60,
                gemini_model=gemini_model,
            )

        payload = build_case_payload(case, state, summary_df)
        all_payloads.append(payload)

        print_case_screen_summary(payload)
        render_case_page(c, payload)

        per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
        summary_df.to_csv(per_case_summary_path, index=False)

        if pause_between_cases and i < len(df_cases):
            input("\nPress Enter to continue to the next case...")

    c.save()

    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path


## Sanity check (optional)

Run this before the example cells if you want a quick check that the core functions are loaded into the current kernel.


In [38]:
print("run_all_categories_live" in globals())
print("run_all_categories_gemini_parent" in globals())
print("run_downstream_planning" in globals())
print("generate_category_activity_bank" in globals())
print("allocate_weekly_slots" in globals())
print("build_weekly_schedule" in globals())
print("summarizer_agent" in globals())


True
True
True
True
True
True
True


## Example run cells

Use **only one** of the cells below at a time.


In [19]:
# Example 1 — one fully live run
# state, summary_df = run_all_categories_live()
# display(summary_df)


In [20]:
# # Example 2 — one synthetic case using Gemini as the parent
# case = df_cases.iloc[0].to_dict()
# state, summary_df = run_all_categories_gemini_parent(
#     case,
#     daily_time_min=10,
#     gemini_model="gemini-2.5-flash",
# )
# display(summary_df)



CHILD REFERENCE CHECK
Name: Noah
Chronological age: 10 months
Diagnosis / condition: Down syndrome
Parent concern: Not sitting independently yet, Hypotonia, slow feeding, socially engaged
Daily time available: 10 minutes

DELAY ESTIMATOR AGENT
Movement / Physical: 4 month starting delay | Hypotonia may impact the child's ability to sit independently.
Social / Emotional: 4 month starting delay | Social engagement is present but may be impacted by hypotonia and developmental delays associated with Down syndrome.
Language / Communication: 6 month starting delay | Down syndrome often impacts language development, leading to a delay.
Cognitive / Adaptive: 6 month starting delay | Cognitive development may be impacted by Down syndrome and associated hypotonia.

MILESTONE Q&A AGENT — Movement / Physical

--- Asking Movement / Physical milestones around 4 months ---
Q: Can Noah bring hands to mouth right now? (yes / sometimes / with_help / no / not_sure)
A (Gemini-parent): sometimes | status:

,child_name,chronological_age_months,diagnosis,concern,daily_time_min,category,estimated_delay_months,delay_reason,estimated_dev_age_months,milestone_gap_months,support_tier,needs_special_support,weekly_target_minutes_for_category,activity_bank_status,activity_bank_summary
0,Noah,10,Down syndrome,"Not sitting independently yet, Hypotonia, slow...",10,Movement / Physical,4,Hypotonia may impact the child's ability to si...,4,6,needs_special_support,True,28,success,This activity bank provides practical home act...
1,Noah,10,Down syndrome,"Not sitting independently yet, Hypotonia, slow...",10,Social / Emotional,4,Social engagement is present but may be impact...,9,1,no_special_support,False,0,no_special_support,We do not think Noah has a meaningful delay in...
2,Noah,10,Down syndrome,"Not sitting independently yet, Hypotonia, slow...",10,Language / Communication,6,Down syndrome often impacts language developme...,6,4,monitor_and_enrich,True,11,success,This activity bank provides practical language...
3,Noah,10,Down syndrome,"Not sitting independently yet, Hypotonia, slow...",10,Cognitive / Adaptive,6,Cognitive development may be impacted by Down ...,6,4,monitor_and_enrich,True,11,success,"This activity bank provides practical, engagin..."


In [ ]:
# Example 3A — export advisor packet in live mode
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export(
#     df_cases=df_cases,
#     mode="live",
#     output_dir="outputs/advisor_case_pack_v4",
#     default_daily_time_min=10,
#     pause_between_cases=True,
# )
# display(advisor_rating_df.head())
# pdf_path


In [ ]:
# Example 3B — export advisor packet in Gemini synthetic-parent mode
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export(
#     df_cases=df_cases,
#     mode="gemini",
#     output_dir="outputs/advisor_case_pack_v4",
#     default_daily_time_min=10,
#     pause_between_cases=False,
#     gemini_model="gemini-2.5-flash",
# )
# display(advisor_rating_df.head())
# pdf_path


In [39]:
import time
from pathlib import Path
from datetime import datetime
import pandas as pd
from reportlab.lib.pagesizes import landscape, LETTER
from reportlab.pdfgen import canvas

def run_cases_and_export_in_batches(
    df_cases: pd.DataFrame,
    batch_size: int = 5,
    gap_between_batches_sec: int = 120,
    mode: str = "gemini",
    output_dir: str = "outputs/advisor_case_pack_v5",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = False,
    gemini_model: str = "gemini-2.5-flash",
):
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))

    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)

    total_cases = len(df_cases)

    for start_idx in range(0, total_cases, batch_size):
        end_idx = min(start_idx + batch_size, total_cases)
        batch_df = df_cases.iloc[start_idx:end_idx]

        print("\n" + "=" * 110)
        print(f"RUNNING BATCH {start_idx // batch_size + 1} | cases {start_idx + 1} to {end_idx} of {total_cases}")
        print("=" * 110)

        for _, row in batch_df.iterrows():
            case = row.to_dict()

            if mode == "live":
                state, summary_df = run_case_live_from_prefilled_case(
                    case=case,
                    default_daily_time_min=default_daily_time_min,
                    min_yes_confirm=2,
                    yes_ratio_confirm=0.60,
                )
            else:
                state, summary_df = run_case_gemini_from_prefilled_case(
                    case=case,
                    daily_time_min=default_daily_time_min,
                    min_yes_confirm=2,
                    yes_ratio_confirm=0.60,
                    gemini_model=gemini_model,
                )

            payload = build_case_payload(case, state, summary_df)
            all_payloads.append(payload)

            print_case_screen_summary(payload)
            render_case_page(c, payload)

            per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
            summary_df.to_csv(per_case_summary_path, index=False)

            if pause_between_cases:
                input("\nPress Enter to continue to the next case...")

        # sleep only between batches, not after the last one
        if end_idx < total_cases:
            print("\n" + "-" * 110)
            print(f"Batch complete. Waiting {gap_between_batches_sec} seconds before next batch...")
            print("-" * 110)
            time.sleep(gap_between_batches_sec)

    c.save()

    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path

In [40]:
# Example 3B — export advisor packet in Gemini synthetic-parent mode, in batches of 5 with 2 min gap
all_payloads, advisor_rating_df, pdf_path = run_cases_and_export_in_batches(
    df_cases=df_cases,
    batch_size=5,
    gap_between_batches_sec=120,   # 2 minutes
    mode="gemini",
    output_dir="outputs/advisor_case_pack_v5",
    default_daily_time_min=10,
    pause_between_cases=False,
    gemini_model="gemini-2.5-flash",
)

display(advisor_rating_df.head())
pdf_path


RUNNING BATCH 1 | cases 1 to 5 of 14

STARTING C01 | Noah | 10 months | Down syndrome
Concerns: Not sitting independently yet, Hypotonia, slow feeding, socially engaged
Daily time: 10 min/day

DELAY ESTIMATOR AGENT
Movement / Physical: 4 month starting delay | Hypotonia and Down syndrome may contribute to delays in independent sitting.
Social / Emotional: 4 month starting delay | Social engagement is present, but developmental delays associated with Down syndrome may impact social skills.
Language / Communication: 6 month starting delay | Down syndrome and hypotonia may impact language development, leading to a delay.
Cognitive / Adaptive: 6 month starting delay | Cognitive development may be impacted by Down syndrome and associated hypotonia.

MILESTONE Q&A AGENT — Movement / Physical

--- Asking Movement / Physical milestones around 4 months ---
Q: Can Noah bring hands to mouth right now? (yes / sometimes / with_help / no / not_sure)
A (Gemini-parent): sometimes | status: ok | reaso

,category,estimated_dev_age_months,gap_months,support_tier,severity
0,Movement / Physical,4,6,needs_special_support,minimal / mild
1,Social / Emotional,9,1,no_special_support,minimal / mild
2,Language / Communication,6,4,monitor_and_enrich,minimal / mild
3,Cognitive / Adaptive,6,4,monitor_and_enrich,minimal / mild



Focus areas:
  • Movement / Physical
  • Language / Communication
  • Cognitive / Adaptive

Daily activities:
1. Tummy Time with Toys | Movement / Physical | 5 min
   Goal: needs_special_support
   Description: Place Noah on his tummy on a soft surface. Place colorful toys just out of reach to encourage him to push up onto his elbows or forearms to reach for them.
2. Sound Imitation Game | Language / Communication | 5 min
   Goal: monitor_and_enrich
   Description: Make different sounds like 'ma', 'ba', and 'ah' and encourage Noah to imitate you. Use exaggerated facial expressions to engage him.
3. Supported Sitting Practice | Movement / Physical | 5 min
   Goal: needs_special_support
   Description: Sit Noah on the floor with pillows around him for support. Encourage him to lean on his hands for balance while you place toys in front of him.
4. Gesture Play | Language / Communication | 5 min
   Goal: monitor_and_enrich
   Description: Use simple gestures like waving or clapping and sa

,category,estimated_dev_age_months,gap_months,support_tier,severity
0,Movement / Physical,18,0,no_special_support,minimal / mild
1,Social / Emotional,18,0,no_special_support,minimal / mild
2,Language / Communication,15,3,monitor_and_enrich,minimal / mild
3,Cognitive / Adaptive,18,0,no_special_support,minimal / mild



Focus areas:
  • Language / Communication

Daily activities:
1. Name That Toy | Language / Communication | 5 min
   Goal: monitor_and_enrich
   Description: Hold up a familiar toy and say its name clearly. Encourage Maya to look at the toy when you name it. You can also ask her to point to the toy when you say its name.
2. Follow the Direction | Language / Communication | 5 min
   Goal: monitor_and_enrich
   Description: Give Maya a simple one-step direction like 'give me the ball' without using gestures. Encourage her to respond by handing you the toy.
3. Sound Imitation Game | Language / Communication | 5 min
   Goal: monitor_and_enrich
   Description: Make simple sounds (like animal noises) and encourage Maya to imitate them. Use animals she knows, like a dog (bark) or cat (meow).
4. Point and Ask | Language / Communication | 7 min
   Goal: monitor_and_enrich
   Description: Place a few toys out of reach. Encourage Maya to point at the toy she wants and say a sound or word. Respond

...


Focus areas:
  • Language / Communication
  • Social / Emotional
  • Cognitive / Adaptive

Daily activities:
1. Body Part Pointing | Language / Communication | 5 min
   Goal: needs_special_support
   Description: Ask Luna to point to different body parts on herself or on a doll. Start with common ones like 'nose' and 'eyes'.
2. Clean Up Time | Social / Emotional | 5 min
   Goal: needs_special_support
   Description: At a designated time, say 'It's clean up time!' and encourage Luna to help pick up her toys. Use eye contact and praise her efforts.
3. Two-Word Phrases | Language / Communication | 5 min
   Goal: needs_special_support
   Description: Encourage Luna to use two-word phrases by modeling them. For example, say 'more juice' and prompt her to repeat it.
4. Look at Me Game | Social / Emotional | 5 min
   Goal: needs_special_support
   Description: Take turns with Luna to show each other something fun you can do. Encourage her to say 'Look at me!' when it's her turn.
5. Gesture G

,category,estimated_dev_age_months,gap_months,support_tier,severity
0,Movement / Physical,36,14,needs_special_support,moderate
1,Social / Emotional,48,2,no_special_support,minimal / mild
2,Language / Communication,48,2,no_special_support,minimal / mild
3,Cognitive / Adaptive,48,2,no_special_support,minimal / mild



Focus areas:
  • Movement / Physical

Daily activities:
1. Dress-Up Challenge | Movement / Physical | 5 min
   Goal: needs_special_support
   Description: Provide Ryan with loose clothing items like a jacket or pants. Encourage him to put them on by himself. Offer assistance as needed, but let him try independently first.
2. Ball Toss | Movement / Physical | 5 min
   Goal: needs_special_support
   Description: Stand a few feet away from Ryan and toss a large ball to him. Encourage him to catch it. If he misses, encourage him to try again. This helps with hand-eye coordination.
3. Bead Stringing | Movement / Physical | 7 min
   Goal: needs_special_support
   Description: Give Ryan large beads or macaroni and a string. Encourage him to string them together, helping him if he struggles. Focus on his grip and coordination.
4. Crayon Drawing | Movement / Physical | 10 min
   Goal: needs_special_support
   Description: Provide Ryan with crayons and paper. Encourage him to hold the crayon be

,case_id,child_name,age_months,diagnosis,clinical_appropriateness_1to5,safety_1to5,practicality_for_parents_1to5,clarity_1to5,overall_usefulness_1to5,what_would_you_change,what_is_missing,any_concerns
0,C01,Noah,10,Down syndrome,,,,,,,,
1,C02,Maya,18,No formal diagnosis,,,,,,,,
2,C03,Leo,26,Autism spectrum disorder,,,,,,,,
3,C04,Sofia,37,Prematurity history,,,,,,,,
4,C05,Ethan,54,Cerebral palsy,,,,,,,,


PosixPath('outputs/advisor_case_pack_v5/genex_case_report_gemini_20260428_135800.pdf')

# How We Designed This Notebook and How the Current Algorithm Works

We built this notebook as a structured prototype of the Genex developmental interview and home-support engine for children ages 0–5. Our goal was to create something that is transparent enough for expert review, flexible enough to improve over time, and practical enough to eventually translate into a real product workflow.

At a high level, the notebook has one shared core pipeline and two different interview modes. The shared core pipeline is:

1. collect or load the child profile  
2. estimate a starting delay by developmental domain  
3. ask milestone questions by domain  
4. convert the answers into an estimated developmental age per domain  
5. determine the level of support needed in each domain  
6. generate activity banks by category  
7. build a realistic weekly schedule based on the parent’s available time  
8. optionally export a one-page case summary and advisor rating sheet  

The two interview modes are:

- **Live parent mode**: we answer the milestone questions directly as the parent or evaluator.
- **Synthetic parent mode**: another language model simulates the parent one question at a time from the child’s profile, while the Genex logic still controls question selection, scoring, and planning.

The point of using both modes is that live mode is better for careful manual review, while synthetic mode is useful for rapid stress-testing, generating benchmark cases, and creating packets for advisor feedback.

## 1. Child profile and intake

The notebook begins by gathering a simple child profile. The core fields are:

- child name
- chronological age in months
- diagnosis or condition, if known
- parent concern in free text
- daily time available for home support activities

We use **months** rather than years because the milestone tables are month-based and developmental changes are more granular in infancy and toddlerhood. This is especially important in younger ages, where a few months can make a meaningful difference.

The `daily_time_min` field is important because we did not want the system to recommend unrealistic activity loads. The weekly schedule is constrained by the parent’s available time rather than assuming unlimited home practice time.

## 2. CDC milestone table loading

The notebook loads a CDC milestone table as the structured source for milestone questions. The table is normalized into a consistent internal format, including:

- `months`
- `category`
- `milestone`
- `category_key`
- `question_id`

We map milestone categories into four working developmental domains:

- Movement / Physical
- Social / Emotional
- Language / Communication
- Cognitive / Adaptive

We use aliases so slightly different category names in the source table can still map into one consistent internal category system.

The reason we use a structured milestone table rather than asking a model to invent milestone questions is to keep the interview grounded, reproducible, and easier to inspect.

## 3. Starting delay estimation by domain

Before asking milestone questions, we estimate a **starting delay** in months for each domain. This is not meant to be a diagnosis and not meant to be the final developmental age. It is only meant to be a **rough starting anchor** to help us decide which age bands to ask first.

The starting delay is estimated separately for each domain using:

- chronological age
- diagnosis / condition
- parent concern
- the domain being evaluated

The output is:

- `delay_months`
- a short reason

The design goal here is to avoid asking all milestones from all ages. Instead, we use the starting delay to center the interview around the most likely developmental range for that specific child and that specific domain.

## 4. How we choose milestone questions

For each domain, we build the milestone question set dynamically.

The main logic is:

- `approx_dev_months = chronological_months - delay_months`
- define a question window around that developmental estimate
- retrieve CDC milestone items in that range
- select the closest age bands
- ask a small number of questions from each band

### Current important parameters

The current synthetic-review setup uses these settings:

- `window_months = 24`
- `target_questions_per_band = 3`
- `max_bands = 3`
- `max_questions_total = 9`
- `ask_limit_per_band = 3`

### Why we chose these numbers

- **24-month window**: wide enough to capture uncertainty without forcing the system to search the full age range every time
- **3 questions per band**: enough to detect a pattern without making the interview too long
- **3 bands**: gives a lower, middle, and upper range around the estimated developmental level
- **9 total questions per category**: keeps synthetic runs and advisor packets feasible
- **3 asked per band**: balances signal with time and cost

## 5. How answers are handled

Every milestone question can be answered using one of these normalized answer types:

- `yes`
- `sometimes`
- `with_help`
- `no`
- `not_sure`

These are also mapped to simple numeric scores internally:

- yes = 1.0
- sometimes = 0.4
- with_help = 0.2
- no = 0.0
- not_sure = 0.1

We use these labels because they are more realistic than forcing a binary yes/no for developmental milestones. Parents often observe partial, inconsistent, or supported performance.

### Live parent mode

In live mode, the parent or evaluator answers directly.

### Synthetic parent mode

In synthetic mode, a different model simulates a parent based on:

- the child profile
- the current domain
- the current milestone question
- a short memory of previous answers for consistency

We added retry logic, pacing between calls, and error handling because synthetic-parent mode is useful for testing but should not contaminate the developmental estimate with API failures. If the model fails, we treat that as an API problem rather than as real developmental uncertainty.

## 6. How developmental age is estimated

After collecting answers within a domain, we summarize them by age band.

For each month band, we count:

- total questions
- number of `yes`
- number of partial responses (`sometimes`, `with_help`, `not_sure`)
- number of `no`
- yes ratio

Then we use a conservative band-confirmation rule.

### Current confirmation parameters

- `min_yes_confirm = 2`
- `yes_ratio_confirm = 0.60`

### Why we chose these numbers

We originally used a stricter threshold, but it was too hard to confirm a band even when the child clearly had several strengths in that range. The current rule means a band is confirmed when it has:

- at least 2 strong yes answers
- and at least 60% yes within that band

This is conservative enough to avoid overestimating developmental age, but not so strict that partial developmental progress is ignored.

### Current developmental age rule

- If one or more bands are confirmed, we use the **highest confirmed band** as the estimated developmental age.
- If no band is fully confirmed but there are partial responses, we anchor to the lowest partial band and may step one band lower.
- If everything is negative, we fall back to the minimum asked band.

## 7. How we decide support level: three tiers

Once we have developmental age and milestone gap for each domain, we assign a support tier.

The three tiers are:

- `needs_special_support`
- `monitor_and_enrich`
- `no_special_support`

We moved to this three-tier system because a strict binary decision was too blunt. In practice, some categories do not need intensive support but still deserve light enrichment or monitoring.

### Inputs to support-tier logic

The support-tier decision uses:

- `chronological_months`
- `dev_age`
- `milestone_gap_months = chrono - dev_age`
- `estimated_delay_months`

### Why we made it age-aware

A 4-month gap means something very different at 10 months than at 60 months. We wanted the system to be more sensitive in infancy and more tolerant of smaller month differences in older children.

So instead of hard-coding diagnoses or using exactly the same month threshold at every age, we use age-scaled thresholds based on the child’s chronological age.

### Why we did not ask GPT again for support tier

We considered making support tier another model decision, but we chose not to. We want support tier to be:

- deterministic
- inspectable
- easier to explain to advisors
- easier to tune after feedback

So GPT helps with the **starting delay estimate** and the **activity generation**, but the support tier itself is based on transparent rules from the developmental results.

## 8. How activities are generated

We intentionally split activity planning into two stages.

### Stage 1: category activity bank

For each domain that needs support or enrichment, the notebook generates a **category-specific activity bank**.

The activity bank is not a day-by-day plan. It is a pool of candidate activities for that category.

Each activity includes fields such as:

- title
- instructions
- duration
- materials
- level (current-level or near-next-step)
- optional extended activity flag

### Why we did it this way

We did **not** want one model call to directly produce a full weekly schedule. That makes it hard to control realism and time budgets.

Instead, we ask the model for a good activity bank for one category. Then we use deterministic scheduling logic to build the weekly plan from those banks.

## 9. How weekly scheduling works

The weekly scheduler uses:

- daily time available
- milestone gap by category
- support tier by category
- category activity banks

The goal is to create a practical plan for a 5-day week.

### Core scheduling logic

1. determine which categories are eligible for support or enrichment  
2. compute gap-based weights  
3. give stronger categories more weekly emphasis  
4. keep the plan within the parent’s daily time budget  
5. rotate categories across the week  
6. ensure the final plan is realistic, short, and home-friendly  

The weekly schedule therefore depends on:
- time available
- severity / gap
- support tier
- what activities exist in each category bank

## 10. Live parent mode vs synthetic parent mode

### Live parent mode is best for:
- manual testing
- debugging specific cases
- discussing logic with advisors
- checking whether the interview feels natural

### Synthetic parent mode is best for:
- batch testing
- creating initial advisor packets
- benchmarking consistency
- finding edge cases

Synthetic parent mode is not meant to replace real parent input in production. It is a testing and review tool.

## 11. How advisor packets are created

For advisor review, we package each case as a 1-page summary with:

1. child profile  
2. parent input (question / answer transcript)  
3. Genex output (developmental estimates and support priorities)  
4. daily activities  
5. summary paragraph  
6. advisor review prompts  

In addition to the PDF case packet, we also create a structured rating sheet with columns such as:

- clinical appropriateness
- safety
- practicality for parents
- clarity
- overall usefulness
- open-text feedback

## 12. What is model-based vs rule-based

### Model-based components
- starting delay estimation by domain
- synthetic parent answers in synthetic mode
- category activity-bank generation

### Rule-based / deterministic components
- CDC table loading and question retrieval
- category mapping
- answer normalization
- band summaries
- developmental-age estimation from band confirmation
- support-tier assignment
- weekly slot allocation
- weekly schedule construction
- export structure

We intentionally kept the state tracking and scheduling logic deterministic because those are the parts we most want to inspect, defend, and improve transparently.

## 13. Why we chose the current parameter values

- **window_months = 24**  
  Wide enough to capture uncertainty, but still focused around the estimated developmental range.

- **target_questions_per_band = 3**  
  Gives enough evidence per band without making the interview too long.

- **max_bands = 3**  
  Lets us sample a lower, middle, and upper zone around the estimated developmental age.

- **max_questions_total = 9**  
  Keeps synthetic runs manageable and reduces API cost / rate-limit risk.

- **ask_limit_per_band = 3**  
  Matches the compact advisor-review design.

- **min_yes_confirm = 2**  
  Prevents overreacting to a single isolated yes.

- **yes_ratio_confirm = 0.60**  
  Slightly conservative, but more flexible than our earlier stricter threshold.

- **delay_between_questions_sec** in synthetic mode  
  Added to reduce rate-limit issues and keep synthetic-parent runs feasible.

- **daily_time_min**  
  Makes the plan realistic for actual families rather than outputting an idealized schedule.

- **3-tier support system**  
  Better than binary support/no-support because it allows lighter enrichment when a domain is not the top concern but still deserves attention.

## 14. What we expect advisors to help us refine

We see this notebook as an inspectable prototype, not a final clinical system. The parts we especially want advisor feedback on are:

- whether the milestone questions chosen feel like the right ones for the child’s level
- whether the developmental-age estimate feels too optimistic or too conservative
- whether the support tiers are set at the right thresholds
- whether the weekly plan gives the right balance across domains
- whether the activities are clinically appropriate and practical for real families
- whether some diagnoses or age groups need different tuning

## 15. How we see this evolving later

Later, in the real app, we do not expect to use a synthetic parent. That is only for testing and advisor review.

Our production direction is:

- real parent answers directly
- local logic handles state tracking and developmental estimation
- retrieval / RAG helps with grounded educational content
- the model helps mainly with:
  - final plan wording
  - doctor visit prep
  - Ask Genex / evidence-grounded Q&A
  - maybe clarifying messy free-text parent concerns

So we see this notebook as the prototype for the developmental interview and home-support planning engine, while the future Genex app will combine this with the retrieval-grounded RAG system we are building separately.

## 16. Summary of the current philosophy

The philosophy of this notebook is:

- use structured milestone references as the backbone
- use models where language flexibility helps
- keep state tracking and planning logic transparent
- prioritize practical home use
- make the system inspectable enough that expert advisors can tell us exactly what to fix

That is why we built it this way: we wanted something we could explain, test, critique, and iteratively improve with expert feedback rather than a black-box output generator.


# Genex Advisor Review Handout  
## Developmental Interview + Home Support Prototype (Ages 0–5)

Thank you for reviewing this early Genex prototype.

This packet contains sample child cases generated through our current developmental interview and home-support planning workflow for ages 0–5. Our goal at this stage is to evaluate whether the logic, developmental estimates, prioritization across domains, and suggested home activities feel clinically reasonable, practical, and safe.

This is still a prototype and is not intended to diagnose or replace clinical judgment. We are using this review process to identify where the system is working well, where it is too aggressive or too conservative, and what should be revised before further development.

## What this prototype is trying to do

This prototype takes a child’s basic profile, asks structured developmental milestone questions, estimates the child’s developmental level across key domains, and produces a realistic home support plan based on the parent’s available time.

The goal is not to diagnose, label, or replace clinical judgment. The goal is to create a transparent, structured first-pass tool that can help families identify priority areas, receive practical home suggestions, and prepare for more informed follow-up with clinicians.

## What information goes into the system

Each case begins with:
- child name
- age in months
- diagnosis or condition, if known
- parent concerns
- daily time available for home support

We use age in **months** rather than years because developmental milestones are much more sensitive to smaller age differences, especially in infancy and toddlerhood.

## What reference framework we use

The milestone interview is grounded in a structured CDC milestone table. We normalize those milestones into four working domains:

- Movement / Physical  
- Social / Emotional  
- Language / Communication  
- Cognitive / Adaptive  

We use the CDC-based milestone table to keep the question set structured, inspectable, and reproducible.

## How the interview works

The system does not ask every milestone at every age. Instead, it first estimates a rough **starting delay** in each domain based on:
- chronological age
- diagnosis / condition
- parent concerns
- the specific developmental domain

This starting delay is only used to decide **where to begin asking questions**.

From there, the system selects milestone questions around the most likely developmental range for that child in that domain. It asks a small set of milestone questions across a few nearby age bands, then summarizes the responses by band.

## How answers are collected

We currently support two modes:

### 1. Live parent mode  
A real person answers the milestone questions directly.

### 2. Synthetic parent mode  
A language model simulates the parent’s answers from the child profile, while the Genex logic still controls:
- which milestones are asked
- how answers are scored
- how developmental age is estimated
- how support is assigned

Synthetic mode is only for internal testing and advisor review. It is not intended to replace real parent input in a production product.

## How developmental age is estimated

For each domain, the notebook groups responses by milestone age band and looks for enough evidence to “confirm” a band. A band is currently treated as confirmed when it has:
- at least 2 strong “yes” responses
- and at least 60% yes answers within that band

If more than one band is confirmed, we use the highest confirmed band as the domain’s estimated developmental age.

## How support is assigned

Once developmental age is estimated in each domain, we compare it with chronological age and assign one of three support tiers:

- **Needs special support**  
- **Monitor and enrich**  
- **No special support**

We moved to this three-tier system because a binary yes/no support decision was too blunt. In practice, some domains do not need intensive support but still deserve light enrichment or monitoring.

We also made the support logic **age-aware**, because the meaning of a 4-month gap is different in a 10-month-old than in a 5-year-old.

## How activities are generated

The activity system works in two steps.

### Step 1: Category activity bank  
For each category that needs support or enrichment, the system builds a category-specific activity bank.

### Step 2: Weekly scheduling  
The system then builds a weekly plan from those category activity banks, using:
- parent daily time available
- support level by domain
- milestone gap by domain
- the activity durations

This helps us create a plan that is more realistic for home use.

## What is model-based vs rule-based

### Model-based components
- rough starting delay estimate by domain
- synthetic parent answers in synthetic mode
- activity-bank wording and activity suggestions

### Rule-based components
- milestone table loading and normalization
- milestone question selection
- answer normalization
- developmental age estimation
- support-tier assignment
- weekly slot allocation
- weekly schedule construction
- case export format

## What we want feedback on

We would especially value feedback on:
- clinical appropriateness
- developmental-age calibration
- support allocation across domains
- practicality for families
- clarity for parents
- safety / caution
- anything important that feels missing

## Current limitations

This is still a prototype and has important limitations:
- it is not diagnostic
- it does not replace developmental or therapeutic assessment
- synthetic-parent mode is only for internal testing
- milestone thresholds and support logic are still being tuned
- some diagnoses and age bands may need different calibration

## Why your feedback matters

Your feedback will directly shape:
- milestone selection logic
- developmental-age calibration
- support-tier thresholds
- activity quality
- report structure
- future doctor-visit preparation and parent-facing product flows

# Short Cover Note

**Dear Advisor,**

Thank you for reviewing this early Genex prototype.

This packet contains sample child cases generated through our current developmental interview and home-support planning workflow for ages 0–5. Our goal at this stage is to evaluate whether the logic, developmental estimates, prioritization across domains, and suggested home activities feel clinically reasonable, practical, and safe.

This is still a prototype and is not intended to diagnose or replace clinical judgment. We are using this review process to identify where the system is working well, where it is too aggressive or too conservative, and what should be revised before further development.

Your feedback on appropriateness, clarity, practicality, safety, and missing elements would be incredibly valuable.

**Best,**  
**Sara / Genex**